# Notebook 3: Constructing the Semantic Network

This is the core mathematical engine of the thesis. The goal here is to take the plot summaries from our clean dataset and map them into a geometric space based on narrative similarity.

We are deliberately keeping this network **plot-centered**. It is not built on star power, directors, or box office returns. We want to see how movies cluster purely based on their story tropes, and *then* map financial success onto those clusters later.

**The Pipeline:**
1. Filter out remaining non-narrative events (like WWE or stand-up specials).
2. Generate 384-dimensional vectors for every plot using SBERT (`all-MiniLM-L6-v2`).
3. Connect the movies using a Mutual K-Nearest Neighbors (KNN) algorithm.
4. Keep only the Largest Connected Component (LCC) to avoid isolated islands.
5. Run Louvain community detection to find natural narrative neighborhoods.
6. Calculate centrality metrics (PageRank, Betweenness) to find core vs. bridge films.

## 1. Install / import packages

In Colab, run the install cell if needed. If packages are already installed, this is harmless.


In [ ]:
import os
import json
import math
import time
import warnings
import networkx as nx

from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors

import shutil
import subprocess
import re
import numpy as np
import pandas as pd
from pathlib import Path


# Check if /content/drive is actually mounted
mounts = subprocess.run(
    ["bash", "-lc", "mount | grep /content/drive || true"],
    capture_output=True,
    text=True
).stdout

print("Current mount info:")
print(mounts if mounts.strip() else "No active /content/drive mount found.")

# If mounted, unmount it safely
if mounts.strip():
    print("Unmounting existing Drive mount...")
    subprocess.run(["bash", "-lc", "fusermount -u /content/drive || umount /content/drive || true"])

# If /content/drive exists but is just a local broken folder, remove it
drive_path = Path("/content/drive")

if drive_path.exists():
    try:
        contents = list(drive_path.iterdir())
        print("Existing /content/drive contents before cleanup:", contents[:10])
    except Exception as e:
        print("Could not list /content/drive:", e)

    # Only remove after unmount attempt
    print("Removing stale /content/drive folder...")
    shutil.rmtree("/content/drive", ignore_errors=True)

# Recreate clean mount point
Path("/content/drive").mkdir(parents=True, exist_ok=True)

print("Clean /content/drive prepared.")

Current mount info:
/content/drive on /content/drive type fuse.drive (rw,nosuid,nodev,relatime,user_id=0,group_id=0)

Unmounting existing Drive mount...
Existing /content/drive contents before cleanup: []
Removing stale /content/drive folder...
Clean /content/drive prepared.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import os

core_dir = Path("/content/drive/MyDrive/thesis_movie_project/processed/core")

print("Core dir:", core_dir)
print("Core dir exists?", core_dir.exists())

if core_dir.exists():
    print("Files inside core dir:")
    for p in core_dir.iterdir():
        print("-", p.name)
else:
    print("Parent exists?", core_dir.parent.exists())
    print("Project exists?", Path("/content/drive/MyDrive/thesis_movie_project").exists())
    print("MyDrive first 30 items:")
    for item in list(Path("/content/drive/MyDrive").iterdir())[:30]:
        print("-", item)

Core dir: /content/drive/MyDrive/thesis_movie_project/processed/core
Core dir exists? True
Files inside core dir:
- df_thesis_films.csv
- df_display_candidates.csv
- df_revenue_analysis.csv
- df_revenue_analysis_enriched.csv
- df_semantic_analysis.csv
- df_roi_analysis.csv


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_movie_project").resolve()

PROCESSED_CORE_DIR = PROJECT_DIR / "processed" / "core"
PROCESSED_NETWORK_DIR = PROJECT_DIR / "processed" / "network"
FEATURES_DIR = PROJECT_DIR / "processed" / "features"
OUTPUT_TABLES_DIR = PROJECT_DIR / "outputs" / "tables"
OUTPUT_FIGURES_DIR = PROJECT_DIR / "outputs" / "figures"

for d in [PROCESSED_NETWORK_DIR, FEATURES_DIR, OUTPUT_TABLES_DIR, OUTPUT_FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEMANTIC_FILE = PROCESSED_CORE_DIR / "df_semantic_analysis.csv"

EMBEDDINGS_FILE = FEATURES_DIR / "sbert_minilm_embeddings.npy"
EMBEDDING_INDEX_FILE = FEATURES_DIR / "sbert_minilm_embedding_index.csv"

EDGE_LIST_OUT = PROCESSED_NETWORK_DIR / "semantic_network_edges_mutual_knn.csv"
NODE_TABLE_OUT = PROCESSED_NETWORK_DIR / "semantic_network_nodes_lcc.csv"
GRAPHML_OUT = PROCESSED_NETWORK_DIR / "semantic_network_lcc.graphml"
COMMUNITY_SUMMARY_OUT = PROCESSED_NETWORK_DIR / "community_summary.csv"
NETWORK_AUDIT_OUT = OUTPUT_TABLES_DIR / "network_audit.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("SEMANTIC_FILE:", SEMANTIC_FILE)
print("SEMANTIC_FILE exists?", SEMANTIC_FILE.exists())

PROJECT_DIR: /content/drive/MyDrive/thesis_movie_project
SEMANTIC_FILE: /content/drive/MyDrive/thesis_movie_project/processed/core/df_semantic_analysis.csv
SEMANTIC_FILE exists? True


## 2. Project paths and parameters

Make sure `PROJECT_DIR` matches the same project root used in Notebooks 1 and 2.

Do **not** point it to `processed/core`. It must point to the main project folder.


In [ ]:
# Core network parameters

TEXT_COLUMN = "overview"

# SBERT model: compact and strong for semantic similarity.
SBERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Number of nearest neighbors before mutual filtering.
# For 20k films, 15-25 is usually reasonable.
K_NEIGHBORS = 20

# Keep only the largest connected component for centrality/community interpretation.
KEEP_LARGEST_CONNECTED_COMPONENT = True

# Optional: similarity floor after mutual KNN.
# Keep as None first. Mutual KNN already controls graph density.
MIN_EDGE_SIMILARITY = None  # Example: 0.35

# Centrality can be expensive. Betweenness is sampled by default.
BETWEENNESS_SAMPLE_SIZE = 1000
RANDOM_STATE = 42

print("Parameters loaded.")
print("K_NEIGHBORS:", K_NEIGHBORS)
print("MIN_EDGE_SIMILARITY:", MIN_EDGE_SIMILARITY)

Parameters loaded.
K_NEIGHBORS: 20
MIN_EDGE_SIMILARITY: None


## 3. Data Loading and Event Quarantine
We load the 20,091 films from the semantic dataset, but before we generate the network, we have to address a critical edge-case.

Raw databases contain filmed events that are technically "feature length" but lack narrative structure (e.g., WWE Wrestlemania, UFC fights, stand-up comedy specials, and live concerts). If left in, these highly repetitive descriptions form dense, artificial black holes in the network.

Using strict regex filtering, we quarantine and remove these non-narrative records. This step successfully dropped roughly 400 events, leaving us with a pristine, thesis-ready corpus of 19,523 true fictional films.

In [ ]:
if not SEMANTIC_FILE.exists():
    raise FileNotFoundError(f"Missing semantic file: {SEMANTIC_FILE}")

df = pd.read_csv(SEMANTIC_FILE)

print("Loaded df:", df.shape)
display(df.head(3))

required_cols = [
    "tmdb_id", "imdb_id", "title", "release_year",
    "primary_genre", "genres_clean", "overview",
    "overview_word_count",
    "runtime", "popularity", "vote_average", "vote_count",
    "revenue_clean", "log_revenue", "cast_clean", "director_name"
]


missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Build embedding text WITHOUT title to avoid title/franchise leakage
df["embedding_text"] = (
    "Genres: " + df["genres_clean"].fillna("").astype(str) +
    ". Plot: " + df["overview"].fillna("").astype(str)
)

# Clean text
df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str).str.strip()

empty_text = (df[TEXT_COLUMN].str.len() == 0).sum()
print("Empty semantic texts:", empty_text)

if empty_text > 0:
    df = df[df[TEXT_COLUMN].str.len() > 0].copy()
    print("After dropping empty texts:", df.shape)

# Graph-specific quarantine: remove non-narrative filmed events

import re

title_text = df["title"].fillna("").astype(str)
overview_text = df["overview"].fillna("").astype(str)
genre_text = df["genres_clean"].fillna("").astype(str)

combined_text = (
    title_text + " " +
    overview_text + " " +
    genre_text
).str.lower()

# These are compact regex strings, not multiline verbose regex.

event_brand_pattern = (
    r"\b("
    r"wwe|wwf|wcw|nxt|aew|tna|impact wrestling|"
    r"wrestlemania|wrestlemania|royal rumble|summerslam|"
    r"survivor series|elimination chamber|money in the bank|"
    r"hell in a cell|clash of champions|crown jewel|"
    r"no mercy|bad blood|backlash|armageddon|judgment day|"
    r"king of the ring|night of champions|"
    r"ufc|bellator|one championship|pride fc|k-1|glory kickboxing"
    r")\b"
)

event_language_pattern = (
    r"\b("
    r"pay[-\s]?per[-\s]?view|main event|championship match|"
    r"title match|cage match|ladder match|tag team|"
    r"wrestling event|professional wrestling|"
    r"mixed martial arts|martial arts event|"
    r"boxing match|boxing event|fight night|"
    r"live event|live broadcast|broadcast live|"
    r"sports event|tournament|final match"
    r")\b"
)

performance_pattern = (
    r"\b("
    r"concert|live at|live from|live in concert|"
    r"stand[-\s]?up|standup special|comedy special|"
    r"awards show|award ceremony|ceremony|pageant|"
    r"stage production|theatre production|theater production|"
    r"opera performance|ballet performance"
    r")\b"
)

template_event_pattern = (
    r"\b("
    r"features matches|features performances|"
    r"superstars compete|fighters compete|teams compete|"
    r"takes place at|held at|emanates from|"
    r"annual event|pay-per-view event"
    r")\b"
)

df["flag_event_brand"] = combined_text.str.contains(
    event_brand_pattern,
    case=False,
    regex=True,
    na=False
)

df["flag_event_language"] = combined_text.str.contains(
    event_language_pattern,
    case=False,
    regex=True,
    na=False
)

df["flag_performance_special"] = combined_text.str.contains(
    performance_pattern,
    case=False,
    regex=True,
    na=False
)

df["flag_event_template"] = combined_text.str.contains(
    template_event_pattern,
    case=False,
    regex=True,
    na=False
)

df["exclude_non_narrative_event"] = (
    df["flag_event_brand"] |
    df["flag_performance_special"] |
    (df["flag_event_language"] & df["flag_event_template"]) |
    (
        df["flag_event_language"] &
        genre_text.str.contains(r"Action|TV Movie|Music", case=False, regex=True, na=False)
    )
)

print("Non-narrative event quarantine counts:")
display(
    df[
        [
            "flag_event_brand",
            "flag_event_language",
            "flag_performance_special",
            "flag_event_template",
            "exclude_non_narrative_event"
        ]
    ].sum().to_frame("n_records")
)

print("Examples to be removed:")
display(
    df.loc[
        df["exclude_non_narrative_event"],
        ["title", "release_year", "primary_genre", "genres_clean", "vote_count", "overview"]
    ].head(50)
)

# Hard diagnostic: if WWE exists before filtering, the filter must catch it.
wwe_before = df["title"].fillna("").str.contains("WWE|WrestleMania|Royal Rumble", case=False, regex=True).sum()
wwe_caught = (
    df["title"].fillna("").str.contains("WWE|WrestleMania|Royal Rumble", case=False, regex=True) &
    df["exclude_non_narrative_event"]
).sum()

print("WWE-like records before filtering:", wwe_before)
print("WWE-like records caught by filter:", wwe_caught)

if wwe_before > 0 and wwe_caught == 0:
    raise ValueError("The event filter is not catching WWE-like records. Stop and inspect regex.")

df = df[~df["exclude_non_narrative_event"]].copy()

# Final hard removal of remaining wrestling/event records

hard_event_pattern = (
    r"wwe|wwf|wcw|nxt|aew|tna|impact wrestling|"
    r"wrestlemania|royal rumble|summerslam|survivor series|"
    r"elimination chamber|money in the bank|hell in a cell|"
    r"clash of champions|crown jewel|no mercy|bad blood|"
    r"backlash|armageddon|judgment day|king of the ring|"
    r"night of champions|great american bash|battleground|"
    r"extreme rules|fastlane|roadblock|vengeance|"
    r"fully loaded|in your house|taboo tuesday|"
    r"december to dismember|starrcade|slamboree|"
    r"halloween havoc|bash at the beach|souled out|"
    r"superbrawl|uncensored|fall brawl|spring stampede|"
    r"ufc|bellator|one championship|pride fc|k-1|glory kickboxing"
)

hard_event_mask = (
    df["title"].fillna("").astype(str).str.lower().str.contains(hard_event_pattern, regex=True, na=False) |
    df["overview"].fillna("").astype(str).str.lower().str.contains(hard_event_pattern, regex=True, na=False)
)

print("Additional hard event records removed:", hard_event_mask.sum())

if hard_event_mask.sum() > 0:
    display(
        df.loc[
            hard_event_mask,
            ["title", "release_year", "primary_genre", "genres_clean", "vote_count", "overview"]
        ].head(50)
    )

df = df[~hard_event_mask].copy()

# Final confirmation
remaining_event_mask = (
    df["title"].fillna("").astype(str).str.lower().str.contains(hard_event_pattern, regex=True, na=False) |
    df["overview"].fillna("").astype(str).str.lower().str.contains(hard_event_pattern, regex=True, na=False)
)

print("Remaining hard event records after final removal:", remaining_event_mask.sum())

if remaining_event_mask.sum() > 0:
    display(
        df.loc[
            remaining_event_mask,
            ["title", "release_year", "primary_genre", "genres_clean", "vote_count", "overview"]
        ].head(50)
    )
    raise ValueError("Hard event records still remain. Inspect examples above.")

df = df.drop(
    columns=[
        "flag_event_brand",
        "flag_event_language",
        "flag_performance_special",
        "flag_event_template",
        "exclude_non_narrative_event"
    ],
    errors="ignore"
)

df = df.reset_index(drop=True).copy()
df["node_id"] = np.arange(len(df), dtype=int)

print("Final semantic rows for embedding/network after event quarantine:", len(df))

Loaded df: (20091, 17)


,tmdb_id,imdb_id,title,release_year,primary_genre,genres_clean,overview,overview_word_count,text_semantic_plot,runtime,popularity,vote_average,vote_count,revenue_clean,log_revenue,cast_clean,director_name
0,11232,tt0035423,Kate & Leopold,2001,Romance,"Romance, Comedy, Fantasy",When her scientist ex-boyfriend discovers a po...,64,"Kate & Leopold. Genres: Romance, Comedy, Fanta...",118.0,3.9819,6.3,1468.0,76019048.0,18.146495,"Del Pentecost, Andrea Barnes, Philip Bosco, Br...",James Mangold
1,299782,tt0069049,The Other Side of the Wind,2018,Drama,Drama,At a media-swamped party to celebrate his seve...,36,The Other Side of the Wind. Genres: Drama. Plo...,122.0,1.5325,6.7,230.0,NaN,NaN,"Cybill Shepherd, Curtis Harrington, Eric Sherm...",Orson Welles
2,23508,tt0082328,Embodiment of Evil,2008,Horror,"Horror, Thriller",Released from the Mental Health Wing of São Pa...,59,"Embodiment of Evil. Genres: Horror, Thriller. ...",94.0,2.5073,5.1,42.0,NaN,NaN,"Alessandra Miranda, Elder Fraga, José Celso Ma...",José Mojica Marins


Empty semantic texts: 0


/tmp/ipykernel_29177/3932193679.py:102: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["flag_event_brand"] = combined_text.str.contains(
/tmp/ipykernel_29177/3932193679.py:109: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["flag_event_language"] = combined_text.str.contains(
/tmp/ipykernel_29177/3932193679.py:116: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["flag_performance_special"] = combined_text.str.contains(
/tmp/ipykernel_29177/3932193679.py:123: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["flag_event_template"] = combined_text.str.contains(


Non-narrative event quarantine counts:


...

Examples to be removed:


,title,release_year,primary_genre,genres_clean,vote_count,overview
5,Grizzly II: Revenge,2020,Horror,"Horror, Music, Thriller",52.0,"All hell breaks loose when a giant grizzly, re..."
22,The Adventures of Ford Fairlane,1990,Comedy,"Comedy, Crime, Mystery, Action",229.0,"To the Los Angeles elite, Ford Fairlane is kno..."
35,Betsy's Wedding,1990,Comedy,"Comedy, Romance",50.0,Offbeat fashion student Betsy Hopper and her s...
57,Deathstalker IV: Match of Titans,1991,Action,"Action, Adventure, Fantasy",57.0,What begins as a grand tournament soon turns s...
68,The Ages of Lulu,1990,Drama,Drama,291.0,Fifteen-year-old Lulu has never known any affe...
183,"Initiation: Silent Night, Deadly Night 4",1990,Horror,Horror,90.0,A reporter investigating the bizarre death of ...
243,Black Demons,1991,Horror,Horror,37.0,"Three American college students, Dick, his sis..."
267,The Dark Backward,1991,Comedy,Comedy,40.0,A man pursues stand-up comedy encouraged by hi...
289,Father of the Bride,1991,Comedy,"Comedy, Drama, Romance",1487.0,"George Banks is an ordinary, middle-class man ..."
337,Madonna: Truth or Dare,1991,Music,"Music, Documentary",165.0,"From the rains of Japan, through threats of ar..."


WWE-like records before filtering: 59
WWE-like records caught by filter: 58
Additional hard event records removed: 200


,title,release_year,primary_genre,genres_clean,vote_count,overview
23,Air America,1990,Action,"Action, Adventure, Comedy, Drama",641.0,Air America was the CIA's private airline oper...
75,Flight of the Intruder,1991,Action,"Action, War",130.0,U.S. Navy pilot Lt. Jake Grafton and his bomba...
104,She Shoots Straight,1990,Action,"Action, Crime",31.0,Hong Kong police officer Mina Kao marries Huan...
111,Jacob's Ladder,1990,Drama,"Drama, Mystery, Horror",1870.0,"After returning home from the Vietnam War, vet..."
195,Steel and Lace,1991,Action,"Action, Horror, Science Fiction",28.0,"After her attackers go free, Gaily Morton, the..."
246,Breathing Fire,1991,Action,Action,26.0,"Michael, a Vietnam vet with two kids, pulls of..."
275,Dogfight,1991,Drama,"Drama, Romance",141.0,"In the fall of 1963, Eddie Birdlace is an 18-y..."
292,For the Boys,1991,Drama,"Drama, Music, Comedy, War",80.0,Talented USO entertainer Dixie Leonard and com...
345,McBain,1991,Action,"Action, War, Drama",48.0,Santos attempts to lead a people's revolt in C...
479,Child's Play 3,1991,Horror,"Horror, Thriller",1730.0,Eight years after seemingly destroying the kil...


Remaining hard event records after final removal: 0
Final semantic rows for embedding/network after event quarantine: 19523


In [ ]:
if EMBEDDINGS_FILE.exists():
    EMBEDDINGS_FILE.unlink()
    print("Deleted old embeddings:", EMBEDDINGS_FILE)

if EMBEDDING_INDEX_FILE.exists():
    EMBEDDING_INDEX_FILE.unlink()
    print("Deleted old embedding index:", EMBEDDING_INDEX_FILE)

Deleted old embeddings: /content/drive/MyDrive/thesis_movie_project/processed/features/sbert_minilm_embeddings.npy
Deleted old embedding index: /content/drive/MyDrive/thesis_movie_project/processed/features/sbert_minilm_embedding_index.csv


## 4. SBERT Embeddings (Text to Math)
Here we convert the human language of our plot overviews into geometric coordinates.

*Crucial methodology note:* We deliberately exclude the movie's title from the text passed to the SBERT model. If titles are included, the model will cheat and cluster franchise films (like *Batman Begins* and *The Batman*) purely based on the title match. By removing the title, we force the AI to group them purely because they share a "dark, vigilante, crime" narrative structure.

In [ ]:
def embeddings_match_dataset(index_file: Path, data: pd.DataFrame) -> bool:
    if not index_file.exists():
        return False
    try:
        idx = pd.read_csv(index_file)
    except Exception:
        return False

    if len(idx) != len(data):
        return False

    needed = ["node_id", "tmdb_id", "title", "release_year"]
    if any(c not in idx.columns for c in needed):
        return False

    left = data[needed].astype(str).reset_index(drop=True)
    right = idx[needed].astype(str).reset_index(drop=True)
    return left.equals(right)


use_cached_embeddings = EMBEDDINGS_FILE.exists() and embeddings_match_dataset(EMBEDDING_INDEX_FILE, df)

print("Cached embeddings usable?", use_cached_embeddings)

Cached embeddings usable? False


In [ ]:
import time
if use_cached_embeddings:
    embeddings = np.load(EMBEDDINGS_FILE)
    print("Loaded cached embeddings:", embeddings.shape)
else:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(SBERT_MODEL_NAME)

    texts = df[TEXT_COLUMN].tolist()

    start = time.time()
    embeddings = model.encode(
        texts,
        batch_size=128,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    elapsed = time.time() - start

    print("Created embeddings:", embeddings.shape)
    print(f"Embedding time: {elapsed/60:.2f} minutes")

    np.save(EMBEDDINGS_FILE, embeddings)

    embedding_index = df[["node_id", "tmdb_id", "imdb_id", "title", "release_year"]].copy()
    embedding_index.to_csv(EMBEDDING_INDEX_FILE, index=False)

    print("Saved embeddings:", EMBEDDINGS_FILE)
    print("Saved embedding index:", EMBEDDING_INDEX_FILE)

# Safety normalization
embeddings = normalize(embeddings, norm="l2", axis=1)
print("Embeddings ready:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

Created embeddings: (19523, 384)
Embedding time: 0.35 minutes
Saved embeddings: /content/drive/MyDrive/thesis_movie_project/processed/features/sbert_minilm_embeddings.npy
Saved embedding index: /content/drive/MyDrive/thesis_movie_project/processed/features/sbert_minilm_embedding_index.csv
Embeddings ready: (19523, 384)


## 5. The "Mutual Handshake" Network (KNN)
To build the network, we calculate the Cosine Distance between all the plot vectors. We ask the algorithm to find the Top 20 closest neighbors for every single movie.

However, instead of keeping all of them, we apply a "Mutual Handshake" rule: Movie A and Movie B are only connected if Movie A is in Movie B's Top 20, **and** Movie B is in Movie A's Top 20. This drops weak, one-sided connections and ensures the network is held together by genuine narrative similarities.

In [ ]:
n = len(df)

if K_NEIGHBORS >= n:
    raise ValueError("K_NEIGHBORS must be smaller than number of films.")

# +1 because the closest neighbor is usually the item itself.
knn = NearestNeighbors(
    n_neighbors=K_NEIGHBORS + 1,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

start = time.time()
knn.fit(embeddings)
distances, indices = knn.kneighbors(embeddings)
elapsed = time.time() - start

print("KNN complete.")
print("Distances shape:", distances.shape)
print("Indices shape:", indices.shape)
print(f"KNN time: {elapsed/60:.2f} minutes")

KNN complete.
Distances shape: (19523, 21)
Indices shape: (19523, 21)
KNN time: 0.17 minutes


## 6. Build directed KNN candidate edges

At this stage, edges are directed: film A may list film B as a neighbor even if B does not list A.


In [ ]:
directed_edges = []

for src in range(n):
    for rank in range(1, K_NEIGHBORS + 1):  # skip rank 0/self
        dst = int(indices[src, rank])
        dist = float(distances[src, rank])
        sim = 1.0 - dist

        if src == dst:
            continue

        if MIN_EDGE_SIMILARITY is not None and sim < MIN_EDGE_SIMILARITY:
            continue

        directed_edges.append((src, dst, sim, rank))

directed_edges_df = pd.DataFrame(
    directed_edges,
    columns=["source_node_id", "target_node_id", "cosine_similarity", "knn_rank"]
)

print("Directed KNN candidate edges:", directed_edges_df.shape)
display(directed_edges_df.head())

Directed KNN candidate edges: (390448, 4)


,source_node_id,target_node_id,cosine_similarity,knn_rank
0,0,18429,0.573613,1
1,0,18593,0.568070,2
2,0,17337,0.556321,3
3,0,17114,0.554636,4
4,0,7272,0.544044,5


## 7. The Mutual KNN Filter (The "Handshake")
To ensure our network is built on genuine narrative similarity, we apply a strict mutual filter. An edge is only formed if Movie A is in Movie B's top 20, **AND** Movie B is in Movie A's top 20.

This instantly drops weak, generic connections, shrinking the network down to roughly 86,000 highly verified, mutual edges.

In [ ]:
edge_lookup = set(zip(
    directed_edges_df["source_node_id"].astype(int),
    directed_edges_df["target_node_id"].astype(int)
))

mutual_rows = []

for row in directed_edges_df.itertuples(index=False):
    src = int(row.source_node_id)
    dst = int(row.target_node_id)

    if src < dst and (dst, src) in edge_lookup:
        reverse = directed_edges_df[
            (directed_edges_df["source_node_id"] == dst) &
            (directed_edges_df["target_node_id"] == src)
        ].iloc[0]

        sim_avg = (float(row.cosine_similarity) + float(reverse.cosine_similarity)) / 2

        mutual_rows.append({
            "source_node_id": src,
            "target_node_id": dst,
            "cosine_similarity": sim_avg,
            "source_rank": int(row.knn_rank),
            "target_rank": int(reverse.knn_rank)
        })

edges = pd.DataFrame(mutual_rows)

print("Mutual KNN edges:", edges.shape)
display(edges.head())

if edges.empty:
    raise ValueError("No mutual KNN edges created. Increase K_NEIGHBORS or remove MIN_EDGE_SIMILARITY.")

Mutual KNN edges: (86697, 5)


,source_node_id,target_node_id,cosine_similarity,source_rank,target_rank
0,0,18429,0.573613,1,2
1,0,18593,0.568070,2,9
2,0,17337,0.556321,3,3
3,0,17114,0.554636,4,12
4,0,7272,0.544044,5,19


## 8. Isolating the Main Continent (LCC)
In any large network, tiny islands of highly specific movies will disconnect from the main web and float away. To ensure our structural math is stable, we drop these tiny clusters and keep only the Largest Connected Component (LCC). The resulting "main continent" retains over 98% of our data (19,202 nodes), proving the mutual network is highly cohesive.

In [ ]:
G = nx.Graph()

# Add all nodes first
for row in df.itertuples(index=False):
    G.add_node(
        int(row.node_id),
        tmdb_id=str(row.tmdb_id),
        imdb_id=str(row.imdb_id),
        title=str(row.title),
        release_year=int(row.release_year) if pd.notna(row.release_year) else None,
        primary_genre=str(row.primary_genre),
        vote_count=float(row.vote_count) if pd.notna(row.vote_count) else np.nan,
        vote_average=float(row.vote_average) if pd.notna(row.vote_average) else np.nan,
        popularity=float(row.popularity) if pd.notna(row.popularity) else np.nan
    )

# Add mutual edges
for row in edges.itertuples(index=False):
    G.add_edge(
        int(row.source_node_id),
        int(row.target_node_id),
        weight=float(row.cosine_similarity),
        cosine_similarity=float(row.cosine_similarity),
        source_rank=int(row.source_rank),
        target_rank=int(row.target_rank)
    )

print("Full graph nodes:", G.number_of_nodes())
print("Full graph edges:", G.number_of_edges())
print("Full graph density:", nx.density(G))

components = sorted(nx.connected_components(G), key=len, reverse=True)
component_sizes = [len(c) for c in components]

print("Number of connected components:", len(component_sizes))
print("Top 10 component sizes:", component_sizes[:10])
print("Largest component share:", round(component_sizes[0] / G.number_of_nodes(), 4))

Full graph nodes: 19523
Full graph edges: 86697
Full graph density: 0.00045494951110992355
Number of connected components: 310
Top 10 component sizes: [19202, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Largest component share: 0.9836


In [ ]:
if KEEP_LARGEST_CONNECTED_COMPONENT:
    largest_nodes = components[0]
    G_lcc = G.subgraph(largest_nodes).copy()
else:
    G_lcc = G.copy()

print("LCC graph nodes:", G_lcc.number_of_nodes())
print("LCC graph edges:", G_lcc.number_of_edges())
print("LCC density:", nx.density(G_lcc))
print("Average degree:", np.mean([d for _, d in G_lcc.degree()]))

LCC graph nodes: 19202
LCC graph edges: 86685
LCC density: 0.00047022274910266435
Average degree: 9.028747005520259


## 9. Finding the Neighborhoods (Louvain Communities)
Now that the network is built, we need to find the boundaries of the narrative tropes. We run the Louvain community detection algorithm, which scans the network and groups nodes that are densely connected to one another.

The algorithm natively found 26 raw communities. (Note: These raw clusters are exported here, but they will be manually reviewed, labeled, and slightly consolidated in the next notebook to produce the final human-readable tropes).

In [ ]:
community_method = None

# Use NetworkX Louvain directly.
# This avoids the python-louvain package conflict and keeps the method stable.
communities = nx.community.louvain_communities(
    G_lcc,
    weight="weight",
    seed=RANDOM_STATE,
    resolution=1.0
)

partition = {}

for cid, nodes in enumerate(communities):
    for node in nodes:
        partition[node] = cid

community_method = "networkx_louvain"

nx.set_node_attributes(G_lcc, partition, "community_id")

community_counts = pd.Series(partition).value_counts().sort_values(ascending=False)

print("Community method:", community_method)
print("Number of communities:", community_counts.shape[0])
print("Top 15 community sizes:")
display(community_counts.head(15).to_frame("n_films"))

Community method: networkx_louvain
Number of communities: 26
Top 15 community sizes:


,n_films
14,1792
22,1735
12,1517
2,1411
24,1327
4,1276
23,1031
8,1005
25,977
10,974


## 10. Centrality and structural position metrics

We don't just want to know *which* community a movie belongs to; we want to know its specific role within the network. We calculate a few key network metrics:

* **Degree / PageRank:** How "central" or archetypal a movie is. Highly central movies represent the safest, most generic version of that trope.
* **Betweenness Centrality:** Which movies act as "bridges" connecting two completely different types of stories together (calculated on a sample to optimize compute time).


In [ ]:
print("Computing centrality metrics...")

degree_dict = dict(G_lcc.degree())
weighted_degree_dict = dict(G_lcc.degree(weight="weight"))
degree_centrality_dict = nx.degree_centrality(G_lcc)

try:
    eigenvector_dict = nx.eigenvector_centrality(
        G_lcc,
        max_iter=500,
        weight="weight"
    )
except Exception as e:
    print("Eigenvector centrality failed, using NaN. Reason:", e)
    eigenvector_dict = {node: np.nan for node in G_lcc.nodes()}

pagerank_dict = nx.pagerank(
    G_lcc,
    weight="weight",
    alpha=0.85
)

# Sampled betweenness for speed
k_between = min(BETWEENNESS_SAMPLE_SIZE, G_lcc.number_of_nodes())
betweenness_dict = nx.betweenness_centrality(
    G_lcc,
    k=k_between,
    weight=None,  # unweighted shortest-path bridge measure
    seed=RANDOM_STATE
)

clustering_dict = nx.clustering(
    G_lcc,
    weight="weight"
)

print("Centrality metrics computed.")

Computing centrality metrics...
Centrality metrics computed.


## 11 & 12. Building the Final Network Tables
We compile all of our calculated data—node IDs, Louvain community assignments, and centrality rankings—into our final `nodes_lcc` and `edges_lcc` dataframes. These tables will serve as the upstream foundation for all financial and success mapping in the final chapters.

In [ ]:
nodes_lcc = pd.DataFrame({"node_id": list(G_lcc.nodes())})

base_cols = [
    "node_id", "tmdb_id", "imdb_id", "title", "release_year",
    "primary_genre", "genres_clean", "overview_word_count",
    "runtime", "popularity", "vote_average", "vote_count",
    "revenue_clean", "log_revenue", "cast_clean", "director_name"
]

nodes_lcc = nodes_lcc.merge(
    df[base_cols],
    on="node_id",
    how="left"
)

nodes_lcc["community_id"] = nodes_lcc["node_id"].map(partition)
nodes_lcc["degree"] = nodes_lcc["node_id"].map(degree_dict)
nodes_lcc["weighted_degree"] = nodes_lcc["node_id"].map(weighted_degree_dict)
nodes_lcc["degree_centrality"] = nodes_lcc["node_id"].map(degree_centrality_dict)
nodes_lcc["eigenvector_centrality"] = nodes_lcc["node_id"].map(eigenvector_dict)
nodes_lcc["pagerank"] = nodes_lcc["node_id"].map(pagerank_dict)
nodes_lcc["betweenness_centrality_sampled"] = nodes_lcc["node_id"].map(betweenness_dict)
nodes_lcc["clustering_coefficient"] = nodes_lcc["node_id"].map(clustering_dict)

# Useful ranking columns
for metric in [
    "degree", "weighted_degree", "degree_centrality",
    "eigenvector_centrality", "pagerank",
    "betweenness_centrality_sampled"
]:
    nodes_lcc[f"{metric}_rank"] = nodes_lcc[metric].rank(
        ascending=False,
        method="min"
    ).astype(int)

print("nodes_lcc:", nodes_lcc.shape)
display(nodes_lcc.head())

nodes_lcc: (19202, 30)


,node_id,tmdb_id,imdb_id,title,release_year,primary_genre,genres_clean,overview_word_count,runtime,popularity,...,eigenvector_centrality,pagerank,betweenness_centrality_sampled,clustering_coefficient,degree_rank,weighted_degree_rank,degree_centrality_rank,eigenvector_centrality_rank,pagerank_rank,betweenness_centrality_sampled_rank
0,0,11232,tt0035423,Kate & Leopold,2001,Romance,"Romance, Comedy, Fantasy",64,118.0,3.9819,...,3.326597e-06,0.000077,0.000700,0.051451,2284,2832,2284,5585,2621,855
1,1,299782,tt0069049,The Other Side of the Wind,2018,Drama,Drama,36,122.0,1.5325,...,9.714745e-07,0.000085,0.000673,0.053100,3043,4199,3043,10945,1190,987
2,2,23508,tt0082328,Embodiment of Evil,2008,Horror,"Horror, Thriller",59,94.0,2.5073,...,1.858977e-06,0.000050,0.000269,0.000000,8334,9222,8334,8082,10086,6105
3,3,260443,tt0088491,Looney Tunes Spotlight Collection: The Premier...,2004,Animation,"Animation, Comedy, Family",35,207.0,4.3270,...,3.995659e-08,0.000043,0.000013,0.403192,13858,14435,13858,18173,12211,17485
4,4,44281,tt0090665,Halfaouine: Boy of the Terraces,1990,Comedy,"Comedy, Drama",64,98.0,2.7434,...,5.753388e-07,0.000071,0.000197,0.064971,4818,5385,4818,12911,3705,8282


## 12. Build LCC edge table

Only edges whose endpoints are inside the largest connected component are exported.


In [ ]:
lcc_node_set = set(nodes_lcc["node_id"].astype(int))

edges_lcc = edges[
    edges["source_node_id"].isin(lcc_node_set) &
    edges["target_node_id"].isin(lcc_node_set)
].copy()

# Add titles for readability
title_map = df.set_index("node_id")["title"].to_dict()
year_map = df.set_index("node_id")["release_year"].to_dict()

edges_lcc["source_title"] = edges_lcc["source_node_id"].map(title_map)
edges_lcc["target_title"] = edges_lcc["target_node_id"].map(title_map)
edges_lcc["source_year"] = edges_lcc["source_node_id"].map(year_map)
edges_lcc["target_year"] = edges_lcc["target_node_id"].map(year_map)

edges_lcc = edges_lcc[
    [
        "source_node_id", "target_node_id",
        "source_title", "target_title",
        "source_year", "target_year",
        "cosine_similarity", "source_rank", "target_rank"
    ]
].copy()

print("edges_lcc:", edges_lcc.shape)
display(edges_lcc.head())

edges_lcc: (86685, 9)


,source_node_id,target_node_id,source_title,target_title,source_year,target_year,cosine_similarity,source_rank,target_rank
0,0,18429,Kate & Leopold,Frontier,2001,2018,0.573613,1,2
1,0,18593,Kate & Leopold,Village Of The Vampire,2001,2020,0.568070,2,9
2,0,17337,Kate & Leopold,Axcellerator,2001,2019,0.556321,3,3
3,0,17114,Kate & Leopold,Angel,2001,2016,0.554636,4,12
4,0,7272,Kate & Leopold,Castle Freak,2001,2020,0.544044,5,19


## 13 & 14. Community Summaries and Network Audit
Before moving on, we generate a high-level summary of the 26 communities, noting their size, top genres, and median financial revenue. We also generate a strict audit table logging the exact parameters (K=20, 19,202 LCC nodes, density) to ensure the methodology can be perfectly reproduced by the thesis committee.

In [ ]:
community_summary = (
    nodes_lcc
    .groupby("community_id")
    .agg(
        n_films=("node_id", "count"),
        median_year=("release_year", "median"),
        min_year=("release_year", "min"),
        max_year=("release_year", "max"),
        median_vote_count=("vote_count", "median"),
        median_vote_average=("vote_average", "median"),
        median_revenue=("revenue_clean", "median"),
        top_genre=("primary_genre", lambda x: x.value_counts().index[0] if len(x.dropna()) else np.nan)
    )
    .reset_index()
    .sort_values("n_films", ascending=False)
)

# Add representative films by weighted degree inside community
representatives = []

for cid, group in nodes_lcc.groupby("community_id"):
    top = group.sort_values("weighted_degree", ascending=False).head(8)
    reps = "; ".join([f"{r.title} ({int(r.release_year)})" for r in top.itertuples(index=False)])
    representatives.append({"community_id": cid, "representative_films": reps})

representatives = pd.DataFrame(representatives)
community_summary = community_summary.merge(representatives, on="community_id", how="left")

print("community_summary:", community_summary.shape)
display(community_summary.head(15))

community_summary: (26, 10)


,community_id,n_films,median_year,min_year,max_year,median_vote_count,median_vote_average,median_revenue,top_genre,representative_films
0,14,1792,2014.0,1990,2024,82.0,6.100,4216409.0,Drama,The House (2024); The Kindred (2021); Pistas p...
1,22,1735,2014.0,1990,2024,79.0,6.300,3014802.0,Drama,Amer (2009); Something New (2016); Mayday (202...
2,12,1517,2013.0,1990,2024,61.0,6.600,7352642.0,Drama,Demon Slayer: Kimetsu no Yaiba - Sibling's Bon...
3,2,1411,2012.0,1990,2024,102.0,6.000,7680455.5,Drama,Amanda (2018); Verbo (2011); Rara (2016); The ...
4,24,1327,2014.0,1990,2024,99.0,6.100,4881639.5,Drama,What's in a Name (2012); A Family (2017); The ...
5,4,1276,2013.0,1990,2024,86.5,6.279,5218006.5,Drama,Sobibor (2018); Europa Europa (1990); Allies (...
6,23,1031,2011.0,1990,2024,123.0,6.100,11782282.0,Action,Living & Dying (2007); 7 Minutes (2014); Rabid...
7,8,1005,2013.0,1990,2024,100.0,5.800,9120108.5,Horror,Haunt (2019); Death's Roulette (2023); Talon F...
8,25,977,2013.0,1990,2024,87.0,6.000,6024477.0,Horror,Crescent City (2024); The Deal (2015); Legenda...
9,10,974,2012.0,1990,2024,128.5,6.100,20136563.0,Animation,Shark Attack 3: Megalodon (2002); All Is Lost ...


## 14. Network audit table

This table should be used in the thesis methodology/results section to describe the graph construction.


In [ ]:
network_audit = pd.DataFrame([
    {"metric": "input_semantic_films", "value": len(df)},
    {"metric": "sbert_model", "value": SBERT_MODEL_NAME},
    {"metric": "k_neighbors", "value": K_NEIGHBORS},
    {"metric": "min_edge_similarity", "value": str(MIN_EDGE_SIMILARITY)},
    {"metric": "directed_knn_edges", "value": len(directed_edges_df)},
    {"metric": "mutual_knn_edges_full_graph", "value": G.number_of_edges()},
    {"metric": "full_graph_nodes", "value": G.number_of_nodes()},
    {"metric": "full_graph_components", "value": len(component_sizes)},
    {"metric": "largest_component_nodes", "value": G_lcc.number_of_nodes()},
    {"metric": "largest_component_edges", "value": G_lcc.number_of_edges()},
    {"metric": "largest_component_share", "value": round(G_lcc.number_of_nodes() / G.number_of_nodes(), 4)},
    {"metric": "average_degree_lcc", "value": round(float(np.mean([d for _, d in G_lcc.degree()])), 4)},
    {"metric": "density_lcc", "value": round(float(nx.density(G_lcc)), 8)},
    {"metric": "community_method", "value": community_method},
    {"metric": "n_communities_lcc", "value": nodes_lcc["community_id"].nunique()},
    {"metric": "betweenness_sample_size", "value": k_between},
])

display(network_audit)

,metric,value
0,input_semantic_films,19523
1,sbert_model,sentence-transformers/all-MiniLM-L6-v2
2,k_neighbors,20
3,min_edge_similarity,None
4,directed_knn_edges,390448
5,mutual_knn_edges_full_graph,86697
6,full_graph_nodes,19523
7,full_graph_components,310
8,largest_component_nodes,19202
9,largest_component_edges,86685


## 15, 16, & 17. Final Exports and Sanity Checks
We export the final network tables, the community summaries, and the GraphML file to our processed folder. A quick printout of the highest-degree nodes (e.g., *Scooby-Doo*, *The House*) confirms the similarity logic worked as intended.

**Next Phase:** The network is officially built. In Notebook 4, we will manually inspect, label, and interpret these 26 communities to turn them into human-readable narrative tropes.

In [ ]:
nodes_lcc.to_csv(NODE_TABLE_OUT, index=False)
edges_lcc.to_csv(EDGE_LIST_OUT, index=False)
community_summary.to_csv(COMMUNITY_SUMMARY_OUT, index=False)
network_audit.to_csv(NETWORK_AUDIT_OUT, index=False)

# GraphML sometimes dislikes NaN attributes, so clean them.
G_export = G_lcc.copy()

for node, attrs in G_export.nodes(data=True):
    for k, v in list(attrs.items()):
        if pd.isna(v):
            attrs[k] = ""

for u, v, attrs in G_export.edges(data=True):
    for k, val in list(attrs.items()):
        if pd.isna(val):
            attrs[k] = ""

nx.write_graphml(G_export, GRAPHML_OUT)

print("Saved network outputs:")
print("-", NODE_TABLE_OUT)
print("-", EDGE_LIST_OUT)
print("-", COMMUNITY_SUMMARY_OUT)
print("-", GRAPHML_OUT)
print("-", NETWORK_AUDIT_OUT)

Saved network outputs:
- /content/drive/MyDrive/thesis_movie_project/processed/network/semantic_network_nodes_lcc.csv
- /content/drive/MyDrive/thesis_movie_project/processed/network/semantic_network_edges_mutual_knn.csv
- /content/drive/MyDrive/thesis_movie_project/processed/network/community_summary.csv
- /content/drive/MyDrive/thesis_movie_project/processed/network/semantic_network_lcc.graphml
- /content/drive/MyDrive/thesis_movie_project/outputs/tables/network_audit.csv


## 16. Quick sanity checks

These are not final interpretation yet. They only tell us whether the graph appears usable.


In [ ]:
print("Top films by weighted degree:")
display(
    nodes_lcc
    .sort_values("weighted_degree", ascending=False)
    [["title", "release_year", "primary_genre", "vote_count", "weighted_degree", "community_id"]]
    .head(20)
)

print("Top films by betweenness:")
display(
    nodes_lcc
    .sort_values("betweenness_centrality_sampled", ascending=False)
    [["title", "release_year", "primary_genre", "vote_count", "betweenness_centrality_sampled", "community_id"]]
    .head(20)
)

print("Largest communities:")
display(community_summary.head(20))

Top films by weighted degree:


,title,release_year,primary_genre,vote_count,weighted_degree,community_id
14843,Scooby-Doo's Creepiest Capers,2000,Animation,37.0,14.478587,21
10443,Scooby-Doo! Curse of the Lake Monster,2010,Mystery,527.0,13.918640,21
3301,Scooby-Doo,2002,Mystery,4744.0,13.827167,21
18274,The House,2024,Drama,41.0,13.276865,14
6824,The Kindred,2021,Horror,125.0,13.124843,14
9388,Amer,2009,Thriller,169.0,13.117239,22
2453,Scooby-Doo on Zombie Island,1998,Animation,831.0,13.116930,21
14187,Scoob!,2020,Animation,1800.0,13.114113,21
11462,Crescent City,2024,Crime,73.0,13.036527,25
15417,Pistas para volver a casa,2015,Mystery,26.0,13.012957,14


Top films by betweenness:


,title,release_year,primary_genre,vote_count,betweenness_centrality_sampled,community_id
1466,The Bandit,1996,Crime,355.0,0.002223,22
5489,The TV Set,2007,Comedy,89.0,0.002018,6
1131,Ace Ventura: When Nature Calls,1995,Crime,4524.0,0.001957,10
12323,Tom and Jerry: Snowman's Land,2022,Animation,49.0,0.001885,10
2897,Intacto,2001,Action,196.0,0.001868,23
352,The Perfect Weapon,1991,Action,124.0,0.001788,10
5668,"Mommy, I'm Scared",2006,Drama,34.0,0.001665,12
16101,Justin Timberlake + The Tennessee Kids,2016,Music,49.0,0.001658,5
13742,Nightcrawler,2014,Crime,11648.0,0.001616,1
10348,Swerve,2012,Crime,47.0,0.001611,23


Largest communities:


,community_id,n_films,median_year,min_year,max_year,median_vote_count,median_vote_average,median_revenue,top_genre,representative_films
0,14,1792,2014.0,1990,2024,82.0,6.1000,4216409.0,Drama,The House (2024); The Kindred (2021); Pistas p...
1,22,1735,2014.0,1990,2024,79.0,6.3000,3014802.0,Drama,Amer (2009); Something New (2016); Mayday (202...
2,12,1517,2013.0,1990,2024,61.0,6.6000,7352642.0,Drama,Demon Slayer: Kimetsu no Yaiba - Sibling's Bon...
3,2,1411,2012.0,1990,2024,102.0,6.0000,7680455.5,Drama,Amanda (2018); Verbo (2011); Rara (2016); The ...
4,24,1327,2014.0,1990,2024,99.0,6.1000,4881639.5,Drama,What's in a Name (2012); A Family (2017); The ...
5,4,1276,2013.0,1990,2024,86.5,6.2790,5218006.5,Drama,Sobibor (2018); Europa Europa (1990); Allies (...
6,23,1031,2011.0,1990,2024,123.0,6.1000,11782282.0,Action,Living & Dying (2007); 7 Minutes (2014); Rabid...
7,8,1005,2013.0,1990,2024,100.0,5.8000,9120108.5,Horror,Haunt (2019); Death's Roulette (2023); Talon F...
8,25,977,2013.0,1990,2024,87.0,6.0000,6024477.0,Horror,Crescent City (2024); The Deal (2015); Legenda...
9,10,974,2012.0,1990,2024,128.5,6.1000,20136563.0,Animation,Shark Attack 3: Megalodon (2002); All Is Lost ...


## 17. Notes for next notebook

The next notebook should be:

`04_network_interpretation.ipynb`

It should use:

- `semantic_network_nodes_lcc.csv`
- `semantic_network_edges_mutual_knn.csv`
- `community_summary.csv`

Main tasks:

1. Interpret major communities.
2. Inspect representative films.
3. Compare central / bridging / peripheral films.
4. Create thesis tables using the display rule, for example `vote_count >= 50`.
5. Prepare figures for the network chapter.
